# Debug camera and robot coordinate frames

**Time:** 90–120 minutes  
**Prerequisites:** notebook 04 and a run containing `camera.json`.

## Learning objectives

- Distinguish pixel, camera, world, robot-base, and object coordinates
- Project a world point into the image and back-project a pixel onto a plane
- Recognize unit, axis, matrix-convention, height, and quaternion mistakes
- Use round-trip tests before touching calibration

> This notebook diagnoses geometry. It does not authorize recalibration. Read `docs/CALIBRATION_V05.md` before changing any camera or tool geometry.

In [ ]:
import json
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

def find_root():
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (p / 'fanuc_er4ia_sim' / 'backproject.py').is_file(): return p
    raise FileNotFoundError('Repository root not found')
ROOT = find_root(); SIM = ROOT / 'fanuc_er4ia_sim'; RUNS = SIM / 'runs'
runs = sorted((p for p in RUNS.glob('*') if (p / 'camera.json').exists()), key=lambda p: p.stat().st_mtime, reverse=True) if RUNS.exists() else []
RUN = runs[0] if runs else RUNS / 'stack_01'
sys.path.insert(0, str(SIM))
from backproject import CameraModel
camera_path = RUN / 'camera.json'
if not camera_path.exists(): raise FileNotFoundError('Capture a run first; camera.json is required')
cam_json = json.loads(camera_path.read_text()); cam = CameraModel(cam_json)
print('Run:', RUN); print('Camera:', cam_json['camera'], cam_json['resolution'])

## 1. Build a mental map

- **Pixels:** `(u, v)` in the image; origin at the upper-left, `u` right, `v` down.
- **Camera:** optical coordinates; USD cameras look along local **−Z**, with +Y up.
- **World:** metres in the USD stage.
- **Robot base:** XY shares the planning world axes. The base origin is at world Z=0.90 m; the calibrated work surface is 5.1 mm below it (robot Z=−0.0051 m).
- **Object:** a pose relative to an individual cube or tool.

The captured USD matrix uses row-vector convention: `[x, y, z, 1] @ M`. The camera basis vectors are matrix rows and the camera position is row 3.

In [ ]:
M = np.array(cam_json['camera_to_world'], dtype=float)
np.set_printoptions(precision=4, suppress=True)
print('camera_to_world =\n', M)
print('eye (world metres):', cam.eye)
print('table world Z:', cam.table_z)
print('basis orthogonality check R @ R.T =\n', cam.R @ cam.R.T)
assert np.allclose(cam.R @ cam.R.T, np.eye(3), atol=1e-5)

## 2. The round-trip test

A geometry pipeline should return to the starting point: world point → pixel → the same world plane. Use known test points, not ground truth copied into a plan.

In [ ]:
test_points = np.array([[0.28, 0.10, cam.table_z], [0.40, 0.00, cam.table_z], [0.50, -0.10, cam.table_z]])
for p in test_points:
    uv = cam.project(p)
    recovered = cam.to_plane(*uv, z=p[2])
    error_mm = np.linalg.norm(recovered - p) * 1000
    print(f'world {p} -> pixel {uv.round(2)} -> world {recovered.round(5)} | error {error_mm:.6f} mm')
    assert error_mm < 1e-5

## 3. Plane height matters

A cube top is 5 mm above the support plane. Intersecting its pixel with the table plane instead of the cube-top plane shifts XY because the ray is angled. This is a common, systematic error—not noise.

In [ ]:
cube_edge = 0.010
cube_centre_xy = np.array([0.36, 0.04])
cube_top = np.r_[cube_centre_xy, cam.table_z + cube_edge]
uv = cam.project(cube_top)
wrong = cam.to_plane(*uv, z=cam.table_z)[:2]
right = cam.to_plane(*uv, z=cam.table_z + cube_edge)[:2]
print('Pixel:', uv.round(2))
print('Assuming table plane:', wrong, 'error mm:', np.linalg.norm(wrong-cube_centre_xy)*1000)
print('Assuming cube-top plane:', right, 'error mm:', np.linalg.norm(right-cube_centre_xy)*1000)

## 4. Visual sanity check

Project a small world grid into the captured image. A believable overlay is not a calibration certificate, but an obviously shifted or mirrored grid quickly exposes wrong axes or the wrong camera file.

In [ ]:
image_path = RUN / 'world_camera.png'
if image_path.exists():
    image = plt.imread(image_path)
    fig, ax = plt.subplots(figsize=(12, 7)); ax.imshow(image)
    for x in np.linspace(0.20, 0.60, 9):
        for y in np.linspace(-0.20, 0.20, 9):
            uv = cam.project([x, y, cam.table_z])
            if uv is not None: ax.plot(uv[0], uv[1], 'c.', markersize=3)
    ax.set_title('World grid projected onto the calibrated work surface'); ax.axis('off'); plt.show()
else: print('world_camera.png is missing')

## 5. Deliberately break it

Predict the symptom before running each experiment:

- Swap `u` and `v`: usually a dramatic diagonal or off-image error.
- Treat metres as millimetres: a 1,000× scale error.
- Use `M[:3, 3]` as the camera position: column-vector thinking applied to a row-vector USD matrix.
- Intersect with the wrong Z plane: systematic XY error that grows away from the optical axis.
- Add half a pixel to continuous coordinates: breaks the exact project/back-project contract.

In [ ]:
p = np.array([0.40, 0.08, cam.table_z])
uv = cam.project(p)
experiments = {
    'correct': cam.to_plane(uv[0], uv[1]),
    'u/v swapped': cam.to_plane(uv[1], uv[0]),
    'half-pixel added': cam.to_plane(uv[0] + 0.5, uv[1] + 0.5),
    'wrong plane +10 mm': cam.to_plane(uv[0], uv[1], z=cam.table_z + 0.010),
}
for name, recovered in experiments.items():
    print(f'{name:20} XY error = {np.linalg.norm(recovered[:2]-p[:2])*1000:8.3f} mm')

## 6. Live Isaac exercise

Create two **new, empty** capture directories using the table and side cameras:

```bash
"$ISAAC/python.sh" fanuc_er4ia_sim/capture_scene.py --out fanuc_er4ia_sim/runs/frame_table --camera table
"$ISAAC/python.sh" fanuc_er4ia_sim/capture_scene.py --out fanuc_er4ia_sim/runs/frame_side --camera side
```

Repeat the grid overlay for both. Explain why the same world point has different pixels but should back-project to the same world location when the correct plane is used.

## Checkpoint

You are finished when you can diagnose these statements without guessing:

- ‘The overlay is mirrored.’ → inspect axis/matrix conventions.
- ‘Error grows with distance from image centre.’ → inspect plane height and intrinsics.
- ‘Every value is about 1,000× wrong.’ → inspect units.
- ‘The math passes but the image is shifted.’ → confirm that `camera.json` and the image came from the same fresh capture.